# Notebook 8: TDI heads on the multitask network

The classification track is the worst-ranked part of this project and every model we have
ever submitted there is gradient boosting on fixed features. The multitask network is our
best model family by a distance, it earned 0.3 to 0.4 weight in the regression blend, and
the TDI track has never seen it.

Fifteen heads: the thirteen regression ones plus `CYP2D6_is_TDI` and `CYP3A4_is_TDI`,
trained jointly so one encoder learns both. That is how the assay works. TDI is defined
as a shift between two IC50 measurements on the same compound, so direct inhibition is
not a separate problem from time-dependent inhibition, it is half of its definition.
Every TDI-labelled row already sits in this table and has a pIC50 label too.

Regression heads use masked squared error on normalised targets; the two TDI heads use
masked binary cross-entropy on raw 0/1 labels, so the loss is right for each.

Where we stand:

| | our shipped OOF AUC | our best measured | a top entry |
| :--- | ---: | ---: | ---: |
| CYP2D6 | 0.5896 | 0.6169 | 0.719 |
| CYP3A4 | 0.8186 | 0.8186 | 0.864 |

About 40 minutes, one seed, resumable per fit. The control is unchanged and still the
gate.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__)
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, time, urllib.request, numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
AUX=np.load(CACHE/'aux_train.npy')
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()

CK='/content/chemprop_medium.pt'
if not Path(CK).exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jeremycheminf/'
        'openadmet_scripts/main/CYP_Challenge/checkpoints/chemprop_medium.pt', CK)
print('checkpoint', round(Path(CK).stat().st_size/1e6,1), 'MB')

URL=('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
     '&record_type=datatable&actvty=all&response_type=save&aid=1851')
if not Path('/content/aid1851.csv').exists():
    urllib.request.urlretrieve(URL, '/content/aid1851.csv')
raw=pd.read_csv('/content/aid1851.csv', low_memory=False, skiprows=[1])
raw['pac50']=-pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
raw=raw.dropna(subset=['panel','pac50','PUBCHEM_EXT_DATASOURCE_SMILES'])
print('qHTS measurements', len(raw))

## Build the 13-column target table

Public compounds that are also challenge compounds are dropped, so nothing the blind set
contains leaks in through an auxiliary head.

In [ ]:
def canon(s):
    m=Chem.MolFromSmiles(s) if isinstance(s,str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours={c for c in map(canon, np.concatenate([SM_TR,SM_TE])) if c}
pub=raw.groupby(['PUBCHEM_EXT_DATASOURCE_SMILES','panel'],as_index=False).pac50.median()
pub['canon']=[canon(s) for s in pub.PUBCHEM_EXT_DATASOURCE_SMILES]
pub=pub.dropna(subset=['canon']); pub=pub[~pub.canon.isin(ours)]
PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
wide=pub.pivot_table(index='PUBCHEM_EXT_DATASOURCE_SMILES',columns='panel',
                     values='pac50').reindex(columns=PAN)
print('public compounds after dropping overlaps', len(wide))

TDI_EP=['CYP2D6','CYP3A4']
HEADS=([f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO]
       + [f'qh_{p}' for p in PAN] + [f'tdi_{e}' for e in TDI_EP])
N_REG=len(HEADS)-len(TDI_EP)
n_ch=len(SM_TR); n_pub=len(wide)
SM_ALL=np.concatenate([SM_TR, wide.index.to_numpy()])
Y=np.full((n_ch+n_pub, len(HEADS)), np.nan, np.float32)
for j,i in enumerate(ISO):
    Y[:n_ch, j]=meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4+j]=AUX[:, j]
Y[n_ch:, 8:13]=wide.to_numpy(float)
for j,e in enumerate(TDI_EP):
    Y[:n_ch, N_REG+j]=meta[f'{e}_is_TDI'].to_numpy(float)
M=np.isfinite(Y)

# Regression heads are normalised on development rows only; the TDI heads stay as raw
# 0/1 because they go through cross-entropy rather than squared error.
_norm=Y.copy(); _norm[np.flatnonzero(hold)]=np.nan
mu=np.nanmean(_norm,0); sd=np.nanstd(_norm,0)
mu[N_REG:]=0.0; sd[N_REG:]=1.0
Yz_nan=((Y-mu)/np.where(sd>0,sd,1.0)).astype(np.float32)
print('table', Y.shape, '| labels per head:', dict(zip(HEADS, M.sum(0).tolist())))

In [ ]:
from chemprop import data, featurizers
from chemprop.models.model import MPNN

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()

# Targets live inside the datapoints, so chemprop's own loader batches them with the
# graphs and nothing has to be re-featurised per minibatch. NaN marks "not measured"
# and becomes the loss mask.
PTS_ALL = [data.MoleculeDatapoint.from_smi(s, Yz_nan[i])
           for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s, np.zeros(len(HEADS), np.float32))
          for s in SM_TE]
DS_TE = data.MoleculeDataset(PTS_TE, FZ)

def loader(idx=None, bs=128, shuffle=False, test=False):
    pts = PTS_TE if test else ([PTS_ALL[i] for i in idx] if idx is not None else PTS_ALL)
    return data.build_dataloader(data.MoleculeDataset(pts, FZ), num_workers=0,
                                 batch_size=bs, shuffle=shuffle, drop_last=False)

def _fp(m, bmg, V_d, X_d):
    # BatchMolGraph.to() moves in place and returns None, unlike a torch tensor, so it
    # must be called as a statement rather than used as an expression.
    if hasattr(bmg, 'to'):
        bmg.to(DEVICE)
    V_d = V_d.to(DEVICE) if torch.is_tensor(V_d) else V_d
    X_d = X_d.to(DEVICE) if torch.is_tensor(X_d) else X_d
    return m.fingerprint(bmg, V_d, X_d)

class Multi(nn.Module):
    def __init__(self, ckpt, n_out, warm=True):
        super().__init__()
        m = MPNN.load_from_file(ckpt)
        if not warm:
            for p in m.parameters():
                nn.init.xavier_uniform_(p) if p.dim() > 1 else nn.init.zeros_(p)
        self.enc = m
        with torch.no_grad():                      # probe on CPU, before .to(DEVICE)
            bmg, V_d, X_d, *_ = next(iter(loader([0, 1, 2, 3], bs=4)))
            d = m.fingerprint(bmg, V_d, X_d).shape[1]
        self.head = nn.Sequential(nn.Linear(d, 512), nn.ReLU(), nn.Dropout(0.1),
                                  nn.Linear(512, n_out))

    def forward(self, bmg, V_d, X_d):
        return self.head(_fp(self.enc, bmg, V_d, X_d))

def train(tr_idx, epochs=25, warm=True, lr=3e-4, bs=128, seed=0):
    torch.manual_seed(seed)
    m = Multi(CK, len(HEADS), warm).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    ld = loader(tr_idx, bs=bs, shuffle=True)
    for ep in range(epochs):
        m.train()
        for batch in ld:
            bmg, V_d, X_d, Yb, *_ = batch
            t = Yb.to(DEVICE).float()
            msk = torch.isfinite(t)
            if msk.sum() == 0:
                continue
            opt.zero_grad()
            p = m(bmg, V_d, X_d)
            tf = torch.nan_to_num(t)
            reg = (((p[:, :N_REG] - tf[:, :N_REG]) ** 2) * msk[:, :N_REG]).sum()
            bce = torch.nn.functional.binary_cross_entropy_with_logits(
                p[:, N_REG:], tf[:, N_REG:], reduction='none')
            loss = (reg + (bce * msk[:, N_REG:]).sum()) / msk.sum()
            loss.backward()
            opt.step()
    return m

@torch.no_grad()
def predict(m, idx=None, bs=256, test=False):
    m.eval()
    out = [m(*batch[:3]).cpu().numpy() for batch in loader(idx, bs=bs, test=test)]
    return np.vstack(out) * sd + mu

## Control first: does the warm start actually load

Two short runs on the same fold, one from the checkpoint and one from a reinitialised
encoder. The warm start should be clearly ahead. If it is not, the checkpoint is not
reaching the weights and everything below is meaningless.

In [ ]:
y1=meta['CYP3A4_pIC50_direct_inhibition'].to_numpy(float)
obs=np.isfinite(y1); dev=np.flatnonzero(obs & ~hold); ho=np.flatnonzero(obs & hold)
pub_idx=np.arange(n_ch, n_ch+n_pub)
quick=np.concatenate([dev[:800], pub_idx[:4000]])
for warm in [True, False]:
    t0=time.time()
    m=train(quick, epochs=6, warm=warm)
    p=predict(m, ho)[:,3]
    print(f'  warm={warm}: CYP3A4 holdout rho {spearmanr(y1[ho],p).statistic:.4f} '
          f'({time.time()-t0:.0f}s for 6 epochs)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

## Full run

Public compounds sit in training for every fold; only the challenge rows rotate.

In [ ]:
EPOCHS = 25
TDI_EP = ['CYP2D6', 'CYP3A4']
import json, pickle
from sklearn.metrics import roc_auc_score, roc_curve

y_tdi = {e: meta[f'{e}_is_TDI'].to_numpy(float) for e in TDI_EP}
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
pub_idx = np.arange(n_ch, len(SM_ALL))
folds = list(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows]))

CKPT = OUT / 'nb8_state.pkl'
acc, done = ({}, set())
if CKPT.exists():
    st = pickle.load(open(CKPT, 'rb')); acc = st['acc']; done = st['done']
    print('resuming;', len(done), 'fits complete')
def save():
    pickle.dump({'acc': acc, 'done': done}, open(CKPT, 'wb'))

sig = lambda x: 1.0 / (1.0 + np.exp(-x))
t0 = time.time()
for f, (tr, va) in enumerate(folds):
    k = f'fold_{f}'
    if k in done: continue
    m = train(np.concatenate([dev_rows[tr], pub_idx]), epochs=EPOCHS)
    acc[k] = predict(m, dev_rows[va]); done.add(k); save()
    print(f'  fold {f} ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()
for k, idx, kw in [('hold', ho_rows, {}), ('test', None, {'test': True})]:
    if k in done: continue
    rows = np.concatenate([dev_rows, pub_idx]) if k == 'hold' else \
           np.concatenate([dev_rows, ho_rows, pub_idx])
    m = train(rows, epochs=EPOCHS)
    acc[k] = predict(m, idx, **kw); done.add(k); save()
    print(f'  {k} model ({time.time()-t0:.0f}s)', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()

oof = np.full((n_ch, len(HEADS)), np.nan, np.float32)
for f, (tr, va) in enumerate(folds):
    oof[dev_rows[va]] = acc[f'fold_{f}']
hp, tp = acc['hold'], acc['test']

def best_mcc(y, p):
    fpr, tpr, _ = roc_curve(y, p); pr = y.mean()
    tp_, fn = tpr * pr, (1 - tpr) * pr
    fp_, tn = fpr * (1 - pr), (1 - fpr) * (1 - pr)
    den = np.sqrt((tp_ + fp_) * (tp_ + fn) * (tn + fp_) * (tn + fn))
    return float(np.nanmax(np.where(den > 0, (tp_ * tn - fp_ * fn) / np.maximum(den, 1e-12), 0)))

RESULTS = []
for j, e in enumerate(TDI_EP):
    col = N_REG + j
    y = y_tdi[e]
    d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    po = sig(oof[d, col]); ph = sig(hp[np.isin(ho_rows, h), col]); pt = sig(tp[:, col])
    np.savez_compressed(OUT / f'tdimtl_{e}.npz', oof=po, dev=d, hold=ph, ho=h, test=pt)
    a, ah = roc_auc_score(y[d], po), roc_auc_score(y[h], ph)
    RESULTS.append(dict(endpoint=e, oof_auc=a, holdout_auc=ah, oof_best_mcc=best_mcc(y[d], po)))
    json.dump(RESULTS, open(OUT / 'results_nb8.json', 'w'), indent=2)
    print(f'{e}: OOF AUC {a:.4f} | holdout AUC {ah:.4f} | best OOF MCC '
          f'{RESULTS[-1]["oof_best_mcc"]:.4f}', flush=True)
print('\nshipped: CYP2D6 0.5896, CYP3A4 0.8186 | our best measured: 0.6169, 0.8186')
print('a top entry:  CYP2D6 0.719,  CYP3A4 0.864')

import shutil
from google.colab import files
shutil.make_archive('/content/notebook8_results', 'zip', OUT)
files.download('/content/notebook8_results.zip')